# Lesson 7 — The Corpus Is the Voice

The same machine, three diets. Everything a model knows comes from its
data — including its blind spots.

In [ ]:
# The corpus: Alice in Wonderland (public domain), with a built-in backup.
import urllib.request, re

FALLBACK = ("the small machine counted every letter of the paragraph and then began "
    "to write its own strange sentences about the city and the lake and the long "
    "quiet train ride home it wrote about the coach and the counselor and the "
    "quiet gym at seven in the morning and although every line was gibberish the "
    "shape of the words was english because the counts had come from english ") * 8

try:
    raw = urllib.request.urlopen("https://www.gutenberg.org/files/11/11-0.txt", timeout=15).read().decode("utf-8")
    raw = raw[raw.find("Alice was beginning"):raw.find("THE END")]
    print("Loaded Alice in Wonderland:", len(raw), "characters")
except Exception as e:
    raw = FALLBACK
    print("Download failed (%s) - using the built-in backup corpus." % type(e).__name__)

# Keep only lowercase letters and spaces - 27 symbols total.
alice = re.sub(r"[^a-z ]+", " ", raw.lower())
alice = re.sub(r" +", " ", corpus).strip()
print("Cleaned corpus:", len(alice), "characters")
print(repr(alice[:100]))

In [ ]:
import re

modern = re.sub(r" +", " ", ("the city council voted on the new bus routes after "
    "residents spoke about long commutes and crowded trains the report showed "
    "that most students ride two buses to school and the board promised new "
    "shelters and better lighting along the main streets by spring ") * 6)

chat = re.sub(r" +", " ", ("lol ok so practice ran late again smh but the scrimmage "
    "was fire ngl we were down bad then came back fr fr coach said run it back "
    "tmrw bring water this time lmao ok gtg dinner see u at the gym bye ") * 6)

corpora = {"alice": alice, "modern": modern, "chat": chat}
for name, c in corpora.items():
    print(f"{name}: {len(c):,} characters")

In [ ]:
import random

def build(c):
    counts = {}
    for i in range(len(c) - 1):
        a, b = c[i], c[i + 1]
        counts.setdefault(a, {}).setdefault(b, 0)
        counts[a][b] += 1
    return counts

def generate(counts, start="t", length=120):
    cur, out = start, start
    for _ in range(length):
        row = counts.get(cur)
        if not row:
            cur = " "
            continue
        cur = random.choices(list(row.keys()), weights=list(row.values()))[0]
        out += cur
    return out

machines = {name: build(c) for name, c in corpora.items()}
for name, m in machines.items():
    print(f"--- trained on {name}")
    print(generate(m))
    print()

Same code ran three times. Every difference came from the data.

## The sharper question: blind spots

What can each machine literally never produce? A letter pair its corpus
never contained has zero odds — forever.

In [ ]:
for name, m in machines.items():
    row = m.get("l", {})
    print(f"{name}: after 'l' it has {len(row)} options; "
          f"'lol' possible: {'o' in row and 'l' in m.get('o', {})}")

# Try your own: pick a pair and check which machines can produce it.

## Turn-in

One generated line from each machine, plus three observations: one voice
difference, one thing only one corpus taught, one blind spot all three
share. End with a sentence about models trained on the whole internet:
whatever is over-represented in the data is over-represented in the odds.